
# Pipeline OCR/VLM - Domiciliation des salaires étrangers

**Modèle : Qwen3.6-VL-27B-FP8 sur GPU Domino**

Migration depuis Qwen2.5-VL-7B-Instruct (v6.7), en alignement avec le
notebook bilans V13c (`bilan_optA`) déjà validé sur ce modèle :

- chargement FP8 via `FineGrainedFP8Config(dequantize=True)` et `device_map='auto'` ;
- classe `AutoModelForImageTextToText` (remplace `AutoModelForVision2Seq`) ;
- `transformers >= 4.57` requis ;
- padding tokenizer à gauche pour le batch ;
- `enable_thinking=False` dans le template de chat (Qwen3) ;
- fenêtre de pixels contrôlée (`min_pixels` / `max_pixels`).

Architecture inchangée :

- classification GPU des pages ;
- extraction spécialisée par type de document ;
- extraction KYC étendue, notamment le père et la mère ;
- conservation des valeurs brutes et normalisées ;
- génération d'un JSON par dossier ;
- export Excel avec les onglets `DOMICILIATIONS`, `PLANNING_TL`, `DOCUMENTS`, `CHAMPS_SOURCE`, `ERREURS` et `PARAMETRES` ;
- gestion des renouvellements et des mois scindés en `P1` / `P2` ;
- checkpoint après chaque dossier pour reprise après incident.

Le pipeline **n'effectue pas les contrôles réglementaires finaux**. Les règles métier et les décisions restent dans Alteryx.


## 1. Dépendances

In [ ]:

# Installation du runtime minimal compatible Qwen3.6-VL-FP8 (cf. bilans V13c)
# %pip install -q -U 'transformers>=4.57.0' accelerate

import sys
from importlib import metadata

REQUIRED_PACKAGES = {
    "torch": "2.0",
    "transformers": "4.57",
    "accelerate": "0.30",
    "PyMuPDF": "1.23",
    "Pillow": "9.0",
    "openpyxl": "3.1",
    "pandas": "1.5",
    "psutil": "5.9",
}

print("Python :", sys.version.replace("\n", " "))
print("\nPackages détectés :")
missing = []
for package_name, minimum in REQUIRED_PACKAGES.items():
    try:
        version = metadata.version(package_name)
        print(f"  {package_name:15s} {version:12s} | minimum conseillé {minimum}")
    except metadata.PackageNotFoundError:
        missing.append(package_name)
        print(f"  {package_name:15s} ABSENT")

if missing:
    raise RuntimeError(
        "Packages manquants : " + ", ".join(missing) +
        ". Installer uniquement ces packages dans l'environnement Domino."
    )

print("\n✅ Vérification des packages terminée sans modification de l'environnement")


## 2. Imports

In [ ]:

import gc
import hashlib
import json
import math
import re
import sys
import time
import calendar
from collections import defaultdict
from datetime import date, datetime, timedelta
from pathlib import Path

import fitz
import numpy as np
import pandas as pd
import psutil
import torch
from PIL import Image
from openpyxl import Workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter
from transformers import AutoProcessor, AutoModelForImageTextToText

print("✅ Imports OK")
print("Python       :", sys.version.split()[0])
print("PyMuPDF     :", fitz.__doc__.splitlines()[0] if fitz.__doc__ else "chargé")
print("Torch       :", torch.__version__)
print("CUDA dispo  :", torch.cuda.is_available())
print("GPU         :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Aucun")


## 3. Configuration

In [ ]:
MODEL_PATH = '/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.6-27B-FP8/main'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
PDF_ZOOM = 3.0
IMAGE_MAX_SIZE = 2024
MIN_PIXELS = 4 * 32 * 32
MAX_PIXELS = 2000 * 32 * 32
BLANK_THRESHOLD = 0.995
GPU_BATCH_SIZE_CLASSIFICATION = 2
GPU_BATCH_SIZE_EXTRACTION = 1
MAX_NEW_TOKENS_CLASSIFICATION = 100
MAX_NEW_TOKENS_EXTRACTION = 1700
INPUT_DIR = Path('/mnt/data/domiciliations_in')
OUTPUT_DIR = Path('/mnt/data/domiciliations_out')
JSON_DIR = OUTPUT_DIR / 'json_dossiers'
LOG_PATH = OUTPUT_DIR / 'pipeline_domiciliations.log'
EXCEL_PATH = OUTPUT_DIR / f"domiciliations_{datetime.now().strftime('%Y%m%d_%H%M')}.xlsx"
MASTER_JSON_PATH = OUTPUT_DIR / 'domiciliations_master.json'
PRORATA_MODE = 'CALENDAR_DAYS'
GENERER_MOIS_COMPLETS = True
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
JSON_DIR.mkdir(parents=True, exist_ok=True)
INPUT_DIR.mkdir(parents=True, exist_ok=True)
pdfs = sorted(INPUT_DIR.glob('*.pdf'))
print(f'Device          : {DEVICE}')
print(f'PDFs détectés   : {len(pdfs)}')
print(f'Entrée          : {INPUT_DIR}')
print(f'Sortie          : {OUTPUT_DIR}')
print(f'Prorata retenu  : {PRORATA_MODE}')
CLASSIFICATION_THRESHOLD = 0.80

PIPELINE_VERSION = "DOM_V7_0_QWEN3_6_VL_27B_FP8"
DOM_REFERENCE_EXCEL = Path("/mnt/data/fichier_domiciliations.xlsx")
PREDOM_REFERENCE_EXCEL = Path("/mnt/data/fichier_predomiciliations.xlsx")
NAME_MATCH_THRESHOLD = 0.86
DATE_TOLERANCE_DAYS = 5
EXCEL_AMOUNT_FORMAT = "0.00"
AMOUNT_FIELDS = {
    "DOM_SALAIRE_NET_MENSUEL", "DOM_PART_TRANSFERABLE",
    "DOM_MONTANT_TOTAL_DOMICILIE", "CTR_SALAIRE_BRUT",
    "CTR_SALAIRE_NET", "CTS_SALAIRE_NET",
    "CTS_PART_TRANSFERABLE", "CTS_PART_PAYABLE_DZD",
}


## 4. Chargement du modèle Qwen3.6-VL-27B-FP8

In [ ]:

if DEVICE != "cuda":
    raise RuntimeError("Ce pipeline nécessite un GPU CUDA.")

torch.backends.cuda.matmul.allow_tf32 = True

print("Chargement du processor...")
t0 = time.time()
processor = AutoProcessor.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
)
processor.tokenizer.padding_side = "left"

print("Chargement du modèle FP8...")
try:
    from transformers.integrations.finegrained_fp8 import FineGrainedFP8Config as FP8Config
except ImportError:
    from transformers import FineGrainedFP8Config as FP8Config

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_PATH,
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    quantization_config=FP8Config(dequantize=True),
)
model.eval()

print(f"✅ Modèle chargé en {time.time() - t0:.1f}s | dtype=bfloat16 (FP8 déquantifié)")
print(f"VRAM allouée : {torch.cuda.memory_allocated() / 1e9:.2f} GB")


## 5. Utilitaires PDF, image et JSON

In [ ]:

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def resize_image(img, max_side=IMAGE_MAX_SIZE):
    w, h = img.size
    if max(w, h) <= max_side:
        return img
    ratio = max_side / max(w, h)
    return img.resize((int(w * ratio), int(h * ratio)), Image.LANCZOS)


def white_ratio(image):
    arr = np.array(image.convert("L"))
    return float((arr > 245).sum() / arr.size)


def is_blank(image, threshold=BLANK_THRESHOLD):
    return white_ratio(image) >= threshold


def pdf_to_pages(path, zoom=PDF_ZOOM):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"PDF introuvable : {path}")
    if path.stat().st_size == 0:
        raise ValueError(f"PDF vide : {path}")

    pages = []
    doc = fitz.open(str(path))
    try:
        page_count = int(doc.page_count)
        if page_count <= 0:
            raise ValueError(f"PyMuPDF ne détecte aucune page dans : {path.name}")

        matrix = fitz.Matrix(zoom, zoom)
        for i in range(page_count):
            page = doc.load_page(i)
            pix = page.get_pixmap(matrix=matrix, alpha=False)

            if pix.width <= 0 or pix.height <= 0 or not pix.samples:
                raise ValueError(
                    f"Rendu image vide : {path.name}, page {i + 1}"
                )

            img = Image.frombytes(
                "RGB",
                (pix.width, pix.height),
                pix.samples
            )
            img = resize_image(img)

            pages.append({
                "index": i,
                "page_num": i + 1,
                "image": img,
                "width": img.width,
                "height": img.height,
                "white_ratio": round(white_ratio(img), 6),
            })
    finally:
        doc.close()

    if len(pages) != page_count:
        raise RuntimeError(
            f"Conversion incomplète de {path.name}: "
            f"{len(pages)} image(s) pour {page_count} page(s)"
        )
    return pages


def parse_json_response(text):
    if not text:
        return {}

    clean = str(text).strip()
    clean = re.sub(r"^```(?:json)?", "", clean, flags=re.I).strip()
    clean = re.sub(r"```$", "", clean).strip()

    match = re.search(r"\{.*\}", clean, flags=re.S)
    if not match:
        return {}

    candidate = match.group(0)
    attempts = [
        candidate,
        re.sub(r",\s*([}\]])", r"\1", candidate),
    ]

    for attempt in attempts:
        try:
            parsed = json.loads(attempt)
            return parsed if isinstance(parsed, dict) else {}
        except Exception:
            continue
    return {}


def log(message):
    line = f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - {message}"
    print(line)
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(line + "\n")


print("✅ Utilitaires PDF/JSON OK")


In [ ]:

def canonical_checkpoint_path(pdf_path):
    """
    Un seul fichier JSON par PDF :
    <nom_du_pdf_sans_extension>.json
    """
    return JSON_DIR / f"{pdf_path.stem}.json"


def checkpoint_is_complete(dossier, pdf_path):
    """
    Un checkpoint est réutilisable seulement s'il correspond au PDF
    et contient une extraction complète avec au moins une page.
    """
    if not isinstance(dossier, dict):
        return False

    stats = dossier.get("stats") or {}
    page_records = dossier.get("page_records") or []

    if dossier.get("source_file") != pdf_path.name:
        return False
    if int(stats.get("pages", 0) or 0) <= 0:
        return False
    if not page_records:
        return False

    # Vérification forte par empreinte SHA-256.
    stored_hash = dossier.get("source_sha256")
    if not stored_hash:
        return False

    try:
        return stored_hash == sha256_file(pdf_path)
    except Exception:
        return False


def load_existing_checkpoint(pdf_path):
    """
    Cherche d'abord le JSON canonique, puis les anciens JSON suffixés
    par l'empreinte. Si un ancien checkpoint valide est trouvé, il est
    migré vers le nom canonique afin de conserver un seul JSON par PDF.
    """
    canonical = canonical_checkpoint_path(pdf_path)
    candidates = [canonical] + sorted(
        JSON_DIR.glob(f"{pdf_path.stem}__*.json"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )

    seen = set()
    for candidate in candidates:
        if candidate in seen or not candidate.exists():
            continue
        seen.add(candidate)

        try:
            dossier = json.loads(candidate.read_text(encoding="utf-8"))
        except Exception:
            continue

        if not checkpoint_is_complete(dossier, pdf_path):
            continue

        # Migration vers un seul JSON canonique.
        if candidate != canonical:
            canonical.write_text(
                json.dumps(dossier, ensure_ascii=False, indent=2, default=str),
                encoding="utf-8",
            )

        # Suppression des anciens doublons JSON du même PDF.
        for duplicate in JSON_DIR.glob(f"{pdf_path.stem}__*.json"):
            if duplicate.exists():
                duplicate.unlink()

        return dossier

    return None


def enrich_dossier_row_with_stats(dossier, statut_traitement):
    """
    Ajoute au niveau dossier les indicateurs visibles dans Excel.
    """
    row = dossier.get("dossier_row") or {}
    stats = dossier.get("stats") or {}

    row.update({
        "STATUT_TRAITEMENT_PIPELINE": statut_traitement,
        "TEMPS_ECOULE_DOSSIER_S": round(float(stats.get("elapsed_s", 0) or 0), 2),
        "TOKENS_IN_DOSSIER": int(stats.get("tokens_in", 0) or 0),
        "TOKENS_OUT_DOSSIER": int(stats.get("tokens_out", 0) or 0),
        "TOKENS_TOTAL_DOSSIER": int(stats.get("tokens_total", 0) or 0),
    })

    dossier["dossier_row"] = row
    return dossier


In [ ]:

def format_duration(seconds):
    seconds = max(0, int(round(float(seconds or 0))))
    hours, rem = divmod(seconds, 3600)
    minutes, secs = divmod(rem, 60)
    if hours:
        return f"{hours:02d}:{minutes:02d}:{secs:02d}"
    return f"{minutes:02d}:{secs:02d}"


def print_pipeline_header(total_pdfs):
    print(
        f"\n{PIPELINE_VERSION} | {total_pdfs} PDF détecté(s)\n",
        flush=True,
    )


def print_compact_progress(
    position,
    total,
    pdf_name,
    pages,
    skipped,
    tokens_in,
    tokens_out,
    elapsed_s,
    pipeline_start,
):
    elapsed_global = time.time() - pipeline_start
    avg = elapsed_global / max(position, 1)
    eta = avg * max(total - position, 0)
    status = "SKIP" if skipped else "TRAITÉ"

    print(
        f"[{position}/{total}] "
        f"{pdf_name} | "
        f"pages={int(pages or 0)} | "
        f"{status} | "
        f"IN={int(tokens_in or 0):,} | "
        f"OUT={int(tokens_out or 0):,} | "
        f"{float(elapsed_s or 0):.2f}s | "
        f"ETA={format_duration(eta)}",
        flush=True,
    )


## 6. Inférence GPU batch

In [ ]:

def apply_template(messages):
    """Qwen3 : désactive le mode 'thinking' si supporté."""
    try:
        return processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        return processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )


def ask_single(prompt, image, max_new_tokens):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt},
        ],
    }]

    text_in = apply_template(messages)
    inputs = processor(
        text=[text_in],
        images=[image],
        return_tensors="pt",
    ).to(DEVICE)

    t0 = time.time()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.0,
            pad_token_id=processor.tokenizer.eos_token_id,
        )
    torch.cuda.synchronize()

    generated = out[0][inputs["input_ids"].shape[1]:]
    text = processor.decode(
        generated,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )

    return {
        "text": text,
        "tokens_in": int(inputs["input_ids"].shape[1]),
        "tokens_out": int(len(generated)),
        "elapsed_s": round(time.time() - t0, 3),
    }


def ask_batch(prompt, images, max_new_tokens):
    if not images:
        return []
    if len(images) == 1:
        return [ask_single(prompt, images[0], max_new_tokens)]

    texts_in = []
    for image in images:
        messages = [{
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }]
        texts_in.append(apply_template(messages))

    inputs = processor(
        text=texts_in,
        images=images,
        return_tensors="pt",
        padding=True,
    ).to(DEVICE)

    t0 = time.time()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.0,
            pad_token_id=processor.tokenizer.eos_token_id,
        )
    torch.cuda.synchronize()

    if out.shape[0] != len(images):
        raise RuntimeError(
            f"Réponses VLM incohérentes : {out.shape[0]} sortie(s) "
            f"pour {len(images)} image(s)"
        )

    elapsed = time.time() - t0
    input_width = inputs["input_ids"].shape[1]
    attention_mask = inputs.get("attention_mask")
    results = []

    for i in range(len(images)):
        generated = out[i][input_width:]
        text = processor.decode(
            generated,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )
        tokens_in = (
            int(attention_mask[i].sum().item())
            if attention_mask is not None
            else int(input_width)
        )
        results.append({
            "text": text,
            "tokens_in": tokens_in,
            "tokens_out": int(len(generated)),
            "elapsed_s": round(elapsed / len(images), 3),
        })

    return results


print("✅ Inférence single/batch OK")


## 7. Prompts de classification et d’extraction

In [ ]:

PROMPT_CLASSIFICATION = """
Analyse uniquement le titre, les en-têtes et la structure générale de cette page.

Classe la page dans exactement une seule catégorie :

- ENGAGEMENT_DOMICILIATION
- CONTRAT_TRAVAIL
- CONTRAT_SPECIFIQUE
- TITRE_TRAVAIL
- PERMIS_TRAVAIL_COUVERTURE
- AUTRE

Règles de classification :
- ENGAGEMENT_DOMICILIATION :
  le titre contient « ENGAGEMENT DE DOMICILIATION »
  ou « CONTRAT DES SALARIES ETRANGERS ».

- CONTRAT_TRAVAIL :
  le titre contient « CONTRAT DE TRAVAIL A DUREE DETERMINEE ».

- CONTRAT_SPECIFIQUE :
  le titre contient « CONTRAT DE TRAVAIL SPECIFIQUE
  A LA MAIN D’OEUVRE ETRANGERE ».

- TITRE_TRAVAIL :
  page bilingue contenant l'identité, le poste, l'employeur,
  les dates et la photo du travailleur.

- PERMIS_TRAVAIL_COUVERTURE :
  page contenant principalement « Permis de Travail »
  et « N° de Série ».

- AUTRE :
  aucun type ne correspond clairement.

Ne te base jamais uniquement sur le numéro de page.

Retourne uniquement ce JSON :
{
  "type_document": "TYPE",
  "confidence": 0.00,
  "titre_detecte": "TITRE BRUT OU null"
}
"""

COMMON_RAW_RULES = """
Tu analyses une seule page.

RÈGLES OBLIGATOIRES :
1. Extraire uniquement les champs demandés.
2. Pour chaque champ, rechercher le libellé indiqué.
3. Recopier uniquement la valeur située juste après le libellé :
   - sur la même ligne ;
   - ou immédiatement sur la ligne suivante si la valeur continue.
4. Conserver la valeur exactement comme elle apparaît :
   espaces, ponctuation, séparateurs, format de date et format de montant.
5. Ne corrige pas l'orthographe.
6. Ne normalise pas les dates.
7. Ne normalise pas les montants.
8. Ne sépare pas automatiquement le nom et le prénom.
9. Ne complète pas une valeur partiellement lisible.
10. N'utilise aucune valeur provenant d'une autre page.
11. Si le libellé est absent ou la valeur illisible, retourne null.
12. N'invente jamais une valeur.
13. Retourne uniquement un objet JSON valide, sans commentaire.
"""

PROMPT_ENGAGEMENT = COMMON_RAW_RULES + """
TYPE ATTENDU : ENGAGEMENT_DOMICILIATION

Extrais exactement les clés suivantes :

{
  "DOM_NOM_RAISON_SOCIAL_CLIENT": null,
  "DOM_COMPTE_LOCAL": null,
  "DOM_ADRESSE_CLIENT": null,
  "DOM_NUMERO_CONTRAT": null,
  "DOM_DUREE_CONTRAT_MOIS": null,
  "DOM_DATE_DEBUT_CONTRAT": null,
  "DOM_DATE_FIN_CONTRAT": null,
  "DOM_NOM_RAISON_SOCIAL_EMPLOYEUR": null,
  "DOM_ADRESSE_EMPLOYEUR": null,
  "DOM_SALAIRE_NET_MENSUEL": null,
  "DOM_PART_TRANSFERABLE": null,
  "DOM_TAUX_TRANSFERABLE": null,
  "DOM_MONTANT_TOTAL_DOMICILIE": null
}

Libellés et règles :

- DOM_NOM_RAISON_SOCIAL_CLIENT :
  valeur après « Nom et raison sociale ».

- DOM_COMPTE_LOCAL :
  valeur après « N de compte » ou « N° de compte ».

- DOM_ADRESSE_CLIENT :
  valeur après la première occurrence de « Adresse »
  dans la section « Identification du client ».

- DOM_NUMERO_CONTRAT :
  valeur après « Numéro du contrat ».

- DOM_DUREE_CONTRAT_MOIS :
  valeur après « Durée du contrat ».

- DOM_DATE_DEBUT_CONTRAT :
  valeur après « Date de début de contrat ».

- DOM_DATE_FIN_CONTRAT :
  valeur après « Date de fin de contrat ».

- DOM_NOM_RAISON_SOCIAL_EMPLOYEUR :
  valeur après « Nom et raison sociale de L’Employeur ».

- DOM_ADRESSE_EMPLOYEUR :
  valeur après « Adresse de L’Employeur ».
  Continuer sur la ligne suivante si l'adresse se poursuit.

- DOM_SALAIRE_NET_MENSUEL :
  valeur après « Salaire net mensuel ».

- DOM_PART_TRANSFERABLE :
  valeur après « Montant de la part transférable ».

- DOM_TAUX_TRANSFERABLE :
  valeur après « Pourcentage en regard du salaire net mensuel ».

- DOM_MONTANT_TOTAL_DOMICILIE :
  valeur après « Montant domicilié en DZD ».
"""

PROMPT_CONTRAT = COMMON_RAW_RULES + """
TYPE ATTENDU : CONTRAT_TRAVAIL

Extrais exactement les clés suivantes :

{
  "CTR_REFERENCE_DOCUMENT": null,
  "CTR_TYPE": null,
  "CTR_EMPLOYEUR": null,
  "CTR_ACTIVITE_EMPLOYEUR": null,
  "CTR_DUREE_MOIS": null,
  "CTR_DATE_DEBUT_CONTRAT": null,
  "CTR_POSTE": null,
  "CTR_NOM_PRENOM_TRAVAILLEUR": null,
  "CTR_PERE_NOM_PRENOM": null,
  "CTR_MERE_NOM_PRENOM": null,
  "CTR_NATIONALITE": null,
  "CTR_DATE_NAISSANCE": null,
  "CTR_LIEU_PAYS_NAISSANCE": null,
  "CTR_ADRESSE_ALGERIE": null,
  "CTR_QUALIFICATION": null,
  "CTR_NUMERO_PERMIS_TRAVAIL": null,
  "CTR_DATE_DELIVRANCE_PERMIS": null,
  "CTR_DATE_DEBUT_VALIDITE_PERMIS": null,
  "CTR_DATE_FIN_VALIDITE_PERMIS": null,
  "CTR_SALAIRE_BRUT": null,
  "CTR_SALAIRE_NET": null,
  "CTR_AFFILIATION_SS": null,
  "CTR_NUMERO_EMPLOYEUR": null,
  "CTR_DATE_SIGNATURE": null,
  "CTR_REFERENCE_DOMICILIATION": null,
  "CTR_SIGNATURE_TRAVAILLEUR_PRESENTE": null,
  "CTR_SIGNATURE_EMPLOYEUR_PRESENTE": null,
  "CTR_CACHET_EMPLOYEUR_PRESENT": null
}

Libellés et règles :

- CTR_REFERENCE_DOCUMENT :
  référence imprimée dans le coin supérieur gauche,
  par exemple « TR-3058 ».

- CTR_TYPE :
  titre complet du document.

- CTR_EMPLOYEUR :
  valeur après
  « au nom de l’employeur ci-après désigné : ».

- CTR_ACTIVITE_EMPLOYEUR :
  valeur après « Nature de l’activité : ».

- CTR_DUREE_MOIS :
  valeur après « pour une durée de : »
  et avant « à compter du ».

- CTR_DATE_DEBUT_CONTRAT :
  valeur après « à compter du : ».

- CTR_POSTE :
  valeur après « En qualité de : ».

- CTR_NOM_PRENOM_TRAVAILLEUR :
  valeur après « A (Mr/Mme) : ».

- CTR_PERE_NOM_PRENOM :
  valeur après « Fils de : »
  et avant « et de : ».

- CTR_MERE_NOM_PRENOM :
  valeur après « et de : ».

- CTR_NATIONALITE :
  valeur après « Nationalité : ».

- CTR_DATE_NAISSANCE :
  valeur après « Né(e) le : »
  et avant « à ».

- CTR_LIEU_PAYS_NAISSANCE :
  valeur après « à » sur la ligne de naissance.

- CTR_ADRESSE_ALGERIE :
  valeur après « Adresse en Algérie : ».

- CTR_QUALIFICATION :
  valeur après « Qualification professionnelle : ».

- CTR_NUMERO_PERMIS_TRAVAIL :
  valeur après « permis de travail N° ».

- CTR_DATE_DELIVRANCE_PERMIS :
  valeur après « Délivré le : ».

- CTR_DATE_DEBUT_VALIDITE_PERMIS :
  première date après « Valable du ».

- CTR_DATE_FIN_VALIDITE_PERMIS :
  date après « au » sur la même ligne.

- CTR_SALAIRE_BRUT :
  valeur après « Montant du salaire mensuel brut : ».

- CTR_SALAIRE_NET :
  valeur après « Montant du salaire mensuel net : ».

- CTR_AFFILIATION_SS :
  valeur après « Affiliation à la sécurité sociale : ».

- CTR_NUMERO_EMPLOYEUR :
  valeur après « Employeur : ».

- CTR_DATE_SIGNATURE :
  date après « Fait à : Bethioua, le ».

- CTR_REFERENCE_DOMICILIATION :
  dans le cachet « DOMICILIATION IMPORT »,
  recopier les cinq cases dans l'ordre
  et les séparer par « | ».
  Exemple : 271901|2026.1|40|00119|DZD

Contrôles visuels :
- CTR_SIGNATURE_TRAVAILLEUR_PRESENTE :
  true si un tracé manuscrit est visible directement sous
  « Signature du Travailleur Etranger », sinon false.

- CTR_SIGNATURE_EMPLOYEUR_PRESENTE :
  true si un tracé manuscrit est visible directement sous
  « Signature de l’Employeur », sinon false.

- CTR_CACHET_EMPLOYEUR_PRESENT :
  true si une empreinte de cachet est visible dans la zone
  « Signature de l’Employeur », sinon false.
"""

PROMPT_CONTRAT_SPECIFIQUE = COMMON_RAW_RULES + """
TYPE ATTENDU : CONTRAT_SPECIFIQUE

Extrais exactement les clés suivantes :

{
  "CTS_REFERENCE_DOCUMENT": null,
  "CTS_EMPLOYEUR": null,
  "CTS_ACTIVITE_EMPLOYEUR": null,
  "CTS_DUREE_MOIS": null,
  "CTS_DATE_DEBUT_CONTRAT": null,
  "CTS_POSTE": null,
  "CTS_NOM_PRENOM_TRAVAILLEUR": null,
  "CTS_PERE_NOM_PRENOM": null,
  "CTS_MERE_NOM_PRENOM": null,
  "CTS_NATIONALITE": null,
  "CTS_DATE_NAISSANCE": null,
  "CTS_LIEU_PAYS_NAISSANCE": null,
  "CTS_ADRESSE_ALGERIE": null,
  "CTS_QUALIFICATION": null,
  "CTS_NUMERO_PERMIS_TRAVAIL": null,
  "CTS_DATE_DELIVRANCE_PERMIS": null,
  "CTS_DATE_DEBUT_VALIDITE_PERMIS": null,
  "CTS_DATE_FIN_VALIDITE_PERMIS": null,
  "CTS_SALAIRE_NET": null,
  "CTS_PART_TRANSFERABLE": null,
  "CTS_PART_PAYABLE_DZD": null,
  "CTS_NUMERO_SS_PAYS_ORIGINE": null,
  "CTS_NUMERO_SS_ALGERIE": null,
  "CTS_DATE_DOCUMENT": null,
  "CTS_SIGNATURE_TRAVAILLEUR_PRESENTE": null,
  "CTS_SIGNATURE_EMPLOYEUR_PRESENTE": null,
  "CTS_CACHET_EMPLOYEUR_PRESENT": null
}

Libellés et règles :

- CTS_REFERENCE_DOCUMENT :
  référence imprimée dans le coin supérieur gauche.

- CTS_EMPLOYEUR :
  valeur après
  « au nom de l’employeur ci-après désigné : ».

- CTS_ACTIVITE_EMPLOYEUR :
  valeur après « Nature de l’activité : ».

- CTS_DUREE_MOIS :
  valeur après « pour une durée de : ».

- CTS_DATE_DEBUT_CONTRAT :
  valeur après « A compter du : ».

- CTS_POSTE :
  valeur après « en qualité de : ».

- CTS_NOM_PRENOM_TRAVAILLEUR :
  valeur après « A (Mr/Mme) : ».

- CTS_PERE_NOM_PRENOM :
  valeur après « Fils de : »
  et avant « et de : ».

- CTS_MERE_NOM_PRENOM :
  valeur après « et de : ».

- CTS_NATIONALITE :
  valeur après « Nationalité : ».

- CTS_DATE_NAISSANCE :
  valeur après « Né(e) le : »
  et avant « à ».

- CTS_LIEU_PAYS_NAISSANCE :
  valeur après « à » sur la ligne de naissance.

- CTS_ADRESSE_ALGERIE :
  valeur après « Adresse en Algérie : ».

- CTS_QUALIFICATION :
  valeur après « Qualification professionnelle : ».

- CTS_NUMERO_PERMIS_TRAVAIL :
  valeur après « permis de travail N° ».

- CTS_DATE_DELIVRANCE_PERMIS :
  valeur après « Délivré le : ».

- CTS_DATE_DEBUT_VALIDITE_PERMIS :
  première date après « valable du ».

- CTS_DATE_FIN_VALIDITE_PERMIS :
  date après « au » sur la même ligne.

- CTS_SALAIRE_NET :
  valeur après « Salaire mensuel de base net : ».

- CTS_PART_TRANSFERABLE :
  valeur après « La part transférable : ».

- CTS_PART_PAYABLE_DZD :
  valeur après « La part payable en dinars algérien : ».

- CTS_NUMERO_SS_PAYS_ORIGINE :
  valeur après « Dans le pays d’origine : ».

- CTS_NUMERO_SS_ALGERIE :
  valeur après « En Algérie : ».

- CTS_DATE_DOCUMENT :
  date après « Fait à : Bethioua, le ».

Contrôles visuels :
- CTS_SIGNATURE_TRAVAILLEUR_PRESENTE :
  true si un tracé manuscrit est visible sous
  « Signature du Travailleur Etranger ».

- CTS_SIGNATURE_EMPLOYEUR_PRESENTE :
  true si un tracé manuscrit est visible sous
  « Signature de l’Employeur ».

- CTS_CACHET_EMPLOYEUR_PRESENT :
  true si une empreinte de cachet est visible dans la zone employeur.
"""

PROMPT_TITRE_TRAVAIL = COMMON_RAW_RULES + """
TYPE ATTENDU : TITRE_TRAVAIL

Le document est bilingue arabe/français.
Utilise les libellés français.

Extrais exactement :

{
  "TTR_TYPE_DOCUMENT": null,
  "TTR_NUMERO": null,
  "TTR_POSTE": null,
  "TTR_DUREE": null,
  "TTR_DATE_DEBUT": null,
  "TTR_DATE_FIN": null,
  "TTR_NOM": null,
  "TTR_PRENOM": null,
  "TTR_DATE_NAISSANCE": null,
  "TTR_LIEU_NAISSANCE": null,
  "TTR_PAYS_NAISSANCE": null,
  "TTR_NATIONALITE": null,
  "TTR_QUALIFICATION": null,
  "TTR_EMPLOYEUR": null,
  "TTR_ADRESSE_EMPLOYEUR": null,
  "TTR_DATE_ENTREE_ALGERIE": null
}

Libellés et règles :

- TTR_TYPE_DOCUMENT :
  titre du document de travail.

- TTR_NUMERO :
  numéro visible dans la zone supérieure du titre,
  uniquement s'il existe un libellé ou un emplacement clairement associé.

- TTR_POSTE :
  valeur dans la zone supérieure gauche correspondant au poste.

- TTR_DUREE :
  valeur après « Durée ».

- TTR_DATE_DEBUT :
  valeur après « Du ».

- TTR_DATE_FIN :
  valeur après « Fin de travail ».

- TTR_NOM :
  valeur après « Nom ».

- TTR_PRENOM :
  valeur après « Prénom ».

- TTR_DATE_NAISSANCE :
  valeur après « Date de naissance ».

- TTR_LIEU_NAISSANCE :
  valeur après « Lieu de naissance ».

- TTR_PAYS_NAISSANCE :
  valeur après « Pays ».

- TTR_NATIONALITE :
  valeur après « Nationalité ».

- TTR_QUALIFICATION :
  valeur après « Qualification ».

- TTR_EMPLOYEUR :
  valeur après « Nom de l’organisme employeur ».

- TTR_ADRESSE_EMPLOYEUR :
  valeur après « Adresse de l’organisme employeur ».

- TTR_DATE_ENTREE_ALGERIE :
  valeur après « Date d’entrée en Algérie ».
"""

PROMPT_PERMIS_COUVERTURE = COMMON_RAW_RULES + """
TYPE ATTENDU : PERMIS_TRAVAIL_COUVERTURE

Extrais exactement :

{
  "PTR_NUMERO_SERIE": null
}

- PTR_NUMERO_SERIE :
  valeur après « N° de Série ».

Ne pas extraire les références légales imprimées à droite.
"""

PROMPTS_EXTRACTION = {
    "ENGAGEMENT_DOMICILIATION": PROMPT_ENGAGEMENT,
    "CONTRAT_TRAVAIL": PROMPT_CONTRAT,
    "CONTRAT_SPECIFIQUE": PROMPT_CONTRAT_SPECIFIQUE,
    "TITRE_TRAVAIL": PROMPT_TITRE_TRAVAIL,
    "PERMIS_TRAVAIL_COUVERTURE": PROMPT_PERMIS_COUVERTURE,
}

TYPES_VALIDES = set(PROMPTS_EXTRACTION) | {"AUTRE"}
print("✅ Prompts V5 bruts spécialisés chargés")


## 8. Normalisation technique

In [ ]:

NULL_VALUES = {"", "NULL", "NONE", "N/A", "NA", "NEANT", "NÉANT", "ILLISIBLE"}

def clean_raw_value(value):
    if value is None or isinstance(value, bool):
        return value
    text = str(value).strip()
    return None if text.upper() in NULL_VALUES else text

def clean_raw_dict(data):
    return {k: clean_raw_value(v) for k, v in data.items()} if isinstance(data, dict) else {}

def normalize_amount(value):
    if value is None:
        return None
    text = re.sub(r"[^0-9,.\-]", "", str(value).replace("\xa0", " ").strip())
    if not text:
        return None
    if "," in text and "." in text:
        text = text.replace(".", "").replace(",", ".") if text.rfind(",") > text.rfind(".") else text.replace(",", "")
    elif "," in text:
        text = text.replace(",", ".")
    try:
        return round(float(text), 2)
    except Exception:
        return None

def parse_date(value):
    if value is None:
        return None
    if isinstance(value, pd.Timestamp):
        return value.date()
    if isinstance(value, datetime):
        return value.date()
    if isinstance(value, date):
        return value
    parsed = pd.to_datetime(str(value).strip(), dayfirst=True, errors="coerce")
    return None if pd.isna(parsed) else parsed.date()

def normalize_text(value):
    if value is None:
        return ""
    text = re.sub(r"[^\w\s]", " ", str(value).upper())
    return re.sub(r"\s+", " ", text).strip()

def name_similarity(a, b):
    from difflib import SequenceMatcher
    a, b = normalize_text(a), normalize_text(b)
    return SequenceMatcher(None, a, b).ratio() if a and b else 0.0


def normalize_dom_reference(value, date_domiciliation=None):
    """
    Format exact : 271901AAAAT40NNNNNDZD
    """
    if value is None:
        return None

    raw = str(value).strip().upper()
    compact = re.sub(r"[^A-Z0-9]", "", raw)

    if re.fullmatch(r"271901\d{4}[1-4]40\d{5}[A-Z]{3}", compact):
        return compact

    parts = [p.strip().upper() for p in re.split(r"[|;/\\]+", raw) if p.strip()]
    if len(parts) >= 5:
        prefix = re.sub(r"\D", "", parts[0])
        fixed_code = re.sub(r"\D", "", parts[2])
        sequence = re.sub(r"\D", "", parts[3]).zfill(5)
        currency = re.sub(r"[^A-Z]", "", parts[4]) or "DZD"
        match_yq = re.search(r"(\d{4})\D*([1-4])", parts[1])
        if prefix == "271901" and fixed_code == "40" and match_yq:
            return f"271901{match_yq.group(1)}{match_yq.group(2)}40{sequence[:5]}{currency[:3]}"

    match_short = re.search(
        r"(\d{4})\D*([1-4])\D*40\D*(\d{1,5})(?:\D*([A-Z]{3}))?",
        raw
    )
    if match_short:
        year = match_short.group(1)
        quarter = match_short.group(2)
        sequence = match_short.group(3).zfill(5)
        currency = match_short.group(4) or "DZD"
        return f"271901{year}{quarter}40{sequence}{currency}"

    return compact or None


def dom_reference_is_valid(value):
    return bool(value and re.fullmatch(r"271901\d{4}[1-4]40\d{5}[A-Z]{3}", str(value)))

def first_not_null(*values):
    return next((v for v in values if v not in (None, "", "NULL")), None)

def safe_read_excel(path):
    path = Path(path)
    if not path.exists():
        return None
    try:
        frames = []
        xls = pd.ExcelFile(path)
        for sheet in xls.sheet_names:
            df = pd.read_excel(path, sheet_name=sheet)
            if not df.empty:
                df["_SOURCE_SHEET"] = sheet
                frames.append(df)
        return pd.concat(frames, ignore_index=True) if frames else None
    except Exception as exc:
        print(f"⚠️ Lecture impossible {path.name}: {exc}")
        return None

def resolve_column(df, aliases):
    if df is None:
        return None
    normalized = {normalize_text(c): c for c in df.columns}
    for alias in aliases:
        if normalize_text(alias) in normalized:
            return normalized[normalize_text(alias)]
    for alias in aliases:
        target = normalize_text(alias)
        for key, original in normalized.items():
            if target in key or key in target:
                return original
    return None

ALIASES = {
    "numero_dom": ["Numéro de domiciliation", "Numero de domiciliation", "N° domiciliation"],
    "date_dom": ["Date domiciliation", "Date de domiciliation", "Date demande", "Request Decision Date"],
    "nom_client": ["Nom complet/Raison social", "Nom complet/Raison sociale", "Nom du Fournisseur/Client", "Name", "Nom"],
    "numero_client": ["Identifiant client", "Numero client", "N° client", "Code client"],
    "date_debut": ["Date début du contrat", "Date debut du contrat"],
    "date_fin": ["Date fin de contrat", "Date de fin du contrat"],
    "reference": ["Référence", "Reference"],
}

def prepare_reference(df, source):
    if df is None or df.empty:
        return None
    out = pd.DataFrame()
    out["SOURCE_MATCH"] = source
    out["SOURCE_SHEET"] = df.get("_SOURCE_SHEET")
    for key, aliases in ALIASES.items():
        col = resolve_column(df, aliases)
        out[key] = df[col] if col else None
    out["numero_dom_normalise"] = out.apply(lambda r: normalize_dom_reference(r.get("numero_dom"), r.get("date_dom")), axis=1)
    out["date_debut_parse"] = out["date_debut"].apply(parse_date)
    out["date_fin_parse"] = out["date_fin"].apply(parse_date)
    return out


def _first_non_empty(series):
    values = [
        v for v in series.tolist()
        if v is not None and not (isinstance(v, float) and pd.isna(v)) and str(v).strip() != ""
    ]
    return values[0] if values else None


def build_reference_table():
    """
    DOM est le référentiel principal.
    PREDOM est joint à gauche uniquement pour enrichir DOM avec :
    - date début du contrat ;
    - date fin du contrat ;
    - référence PREDOM.

    Une ligne DOM + une ligne PREDOM portant le même numéro DOM
    représentent un seul dossier, et non deux candidats.
    """
    dom = prepare_reference(safe_read_excel(DOM_REFERENCE_EXCEL), "DOM")
    predom = prepare_reference(safe_read_excel(PREDOM_REFERENCE_EXCEL), "PREDOM")

    if dom is None or dom.empty:
        return None

    # Une seule ligne principale par numéro DOM.
    # Les doublons DOM restent signalés pour éviter une attribution automatique risquée.
    dom = dom.copy()
    dom["DOM_MATCH_COUNT"] = dom.groupby("numero_dom_normalise")["numero_dom_normalise"].transform("size")

    if predom is None or predom.empty:
        dom["date_debut_predom"] = None
        dom["date_fin_predom"] = None
        dom["reference_predom"] = None
        dom["PREDOM_MATCH_COUNT"] = 0
        return dom

    predom = predom.copy()

    # Consolidation PREDOM par numéro DOM.
    # On ne choisit pas arbitrairement entre plusieurs valeurs différentes :
    # le nombre de lignes est conservé dans PREDOM_MATCH_COUNT.
    predom_grouped = (
        predom.groupby("numero_dom_normalise", dropna=False)
        .agg(
            date_debut_predom=("date_debut", _first_non_empty),
            date_fin_predom=("date_fin", _first_non_empty),
            date_debut_predom_parse=("date_debut_parse", _first_non_empty),
            date_fin_predom_parse=("date_fin_parse", _first_non_empty),
            reference_predom=("reference", _first_non_empty),
            PREDOM_MATCH_COUNT=("numero_dom_normalise", "size"),
        )
        .reset_index()
    )

    reference = dom.merge(
        predom_grouped,
        on="numero_dom_normalise",
        how="left",
        validate="many_to_one",
    )

    reference["PREDOM_MATCH_COUNT"] = (
        reference["PREDOM_MATCH_COUNT"].fillna(0).astype(int)
    )

    # Pour le matching par période, les dates PREDOM sont prioritaires.
    # Si elles sont absentes, on utilise les dates disponibles dans DOM.
    reference["date_debut_match"] = reference["date_debut_predom_parse"].where(
        reference["date_debut_predom_parse"].notna(),
        reference["date_debut_parse"],
    )
    reference["date_fin_match"] = reference["date_fin_predom_parse"].where(
        reference["date_fin_predom_parse"].notna(),
        reference["date_fin_parse"],
    )

    return reference

print("✅ Helpers V6.3 chargés : DOM principal + enrichissement PREDOM")


## 9. Consolidation des documents et informations KYC

In [ ]:

def build_page_row(pdf_name, page_record):
    row = {
        "FICHIER": pdf_name,
        "PAGE": page_record.get("page_num"),
        "TYPE_DOCUMENT": page_record.get("doc_type"),
        "TITRE_DETECTE": page_record.get("titre_detecte"),
        "CONFIANCE_CLASSIFICATION": page_record.get("classification_confidence"),
        "STATUT_EXTRACTION": page_record.get("extraction_status"),
        "ERREUR_EXTRACTION": page_record.get("extraction_error"),
    }
    row.update(page_record.get("raw_data") or {})
    return row

def consolidate_dossier(pdf_name, records):
    row = {
        "FICHIER": pdf_name,
        "NB_PAGES": len(records),
        "TYPES_DOCUMENTS": " | ".join(str(r.get("doc_type")) for r in records),
    }
    pages_by_type = defaultdict(list)
    for record in records:
        pages_by_type[record.get("doc_type")].append(str(record.get("page_num")))
        for key, value in (record.get("raw_data") or {}).items():
            if row.get(key) in (None, ""):
                row[key] = value
    for doc_type, pages in pages_by_type.items():
        row[f"PAGES_{doc_type}"] = ",".join(pages)

    for field in AMOUNT_FIELDS:
        if field in row:
            row[field + "_RAW"] = row[field]
            row[field] = normalize_amount(row[field])

    raw_ref = first_not_null(row.get("CTR_REFERENCE_DOMICILIATION"))
    row["REFERENCE_DOM_EXTRAITE_RAW"] = raw_ref
    row["REFERENCE_DOM_EXTRAITE_NORMALISEE"] = normalize_dom_reference(raw_ref)
    row["REFERENCE_DOM_FORMAT_VALIDE"] = dom_reference_is_valid(row["REFERENCE_DOM_EXTRAITE_NORMALISEE"])
    row["NOM_CLIENT_REFERENCE"] = first_not_null(
        row.get("DOM_NOM_RAISON_SOCIAL_CLIENT"),
        row.get("CTR_NOM_PRENOM_TRAVAILLEUR"),
        row.get("CTS_NOM_PRENOM_TRAVAILLEUR"),
        " ".join(x for x in [str(row.get("TTR_NOM") or "").strip(), str(row.get("TTR_PRENOM") or "").strip()] if x) or None,
    )
    row["NUMERO_CONTRAT_REFERENCE"] = first_not_null(row.get("DOM_NUMERO_CONTRAT"), row.get("CTR_REFERENCE_DOCUMENT"), row.get("CTS_REFERENCE_DOCUMENT"))
    row["DATE_DEBUT_CONTRAT_REFERENCE"] = first_not_null(row.get("DOM_DATE_DEBUT_CONTRAT"), row.get("CTR_DATE_DEBUT_CONTRAT"), row.get("CTS_DATE_DEBUT_CONTRAT"))
    row["DATE_FIN_CONTRAT_REFERENCE"] = first_not_null(row.get("DOM_DATE_FIN_CONTRAT"), row.get("CTR_DATE_FIN_CONTRAT"), row.get("CTS_DATE_FIN_CONTRAT"))
    row["NUMERO_PERMIS_REFERENCE"] = first_not_null(row.get("CTR_NUMERO_PERMIS_TRAVAIL"), row.get("CTS_NUMERO_PERMIS_TRAVAIL"), row.get("TTR_NUMERO"), row.get("PTR_NUMERO_SERIE"))
    return row


def match_dossier(row, ref):
    """
    Matching sécurisé.

    1. Si le numéro DOM est extrait du PDF :
       correspondance exacte uniquement.

    2. Si aucun numéro DOM n'est extrait :
       recherche uniquement sur la période exacte du contrat
       (date début + date fin).

    3. Le numéro DOM n'est retenu que lorsqu'un seul candidat DOM
       est incontestable. Sinon, NUMERO_DOM_RETENU reste vide.
    """
    result = {
        "MATCH_SOURCE": None,
        "MATCH_METHOD": None,
        "MATCH_SCORE": 0.00,
        "MATCH_STATUS": "AUCUN_MATCH",
        "MATCH_CANDIDATES_COUNT": 0,
        "NUMERO_CLIENT_RETENU": None,
        "NUMERO_DOM_RETENU": None,
        "DATE_DOM_RETENUE": None,
        "REFERENCE_EXTERNE_RETENUE": None,
        "REFERENCE_PREDOM_RETENUE": None,
        "DATE_DEBUT_CONTRAT_PREDOM": None,
        "DATE_FIN_CONTRAT_PREDOM": None,
        "PREDOM_TROUVEE": False,
    }

    if ref is None or ref.empty:
        result["MATCH_STATUS"] = "REFERENTIEL_ABSENT"
        return result

    dom_ref = row.get("REFERENCE_DOM_EXTRAITE_NORMALISEE")

    # ---------------------------------------------------------
    # 1) Numéro DOM extrait : matching exact uniquement
    # ---------------------------------------------------------
    if dom_ref:
        exact = ref[ref["numero_dom_normalise"] == dom_ref].copy()
        result["MATCH_CANDIDATES_COUNT"] = int(len(exact))
        result["MATCH_METHOD"] = "NUMERO_DOM_EXACT"

        if len(exact) == 0:
            result["MATCH_STATUS"] = "NUMERO_DOM_NON_TROUVE"
            return result

        if len(exact) > 1 or int(exact.iloc[0].get("DOM_MATCH_COUNT") or 1) > 1:
            result["MATCH_STATUS"] = "PLUSIEURS_CANDIDATS_DOM"
            return result

        best = exact.iloc[0]

    # ---------------------------------------------------------
    # 2) Numéro DOM absent : période exacte et candidat unique
    # ---------------------------------------------------------
    else:
        start_date = parse_date(row.get("DATE_DEBUT_CONTRAT_REFERENCE"))
        end_date = parse_date(row.get("DATE_FIN_CONTRAT_REFERENCE"))

        if not start_date or not end_date:
            result["MATCH_STATUS"] = "DATES_CONTRAT_INSUFFISANTES"
            result["MATCH_METHOD"] = "PERIODE_CONTRAT_EXACTE"
            return result

        period_matches = ref[
            (ref["date_debut_match"] == start_date)
            & (ref["date_fin_match"] == end_date)
        ].copy()

        result["MATCH_CANDIDATES_COUNT"] = int(len(period_matches))
        result["MATCH_METHOD"] = "PERIODE_CONTRAT_EXACTE"

        if len(period_matches) == 0:
            result["MATCH_STATUS"] = "AUCUN_MATCH_PERIODE"
            return result

        if len(period_matches) > 1:
            result["MATCH_STATUS"] = "PLUSIEURS_CANDIDATS_PERIODE"
            return result

        best = period_matches.iloc[0]

        # Sécurité supplémentaire : pas d'attribution si le DOM principal
        # contient lui-même plusieurs lignes pour ce numéro.
        if int(best.get("DOM_MATCH_COUNT") or 1) > 1:
            result["MATCH_STATUS"] = "PLUSIEURS_CANDIDATS_DOM"
            return result

    # ---------------------------------------------------------
    # 3) Attribution seulement après match unique et sûr
    # ---------------------------------------------------------
    predom_count = int(best.get("PREDOM_MATCH_COUNT") or 0)

    result.update({
        "MATCH_SOURCE": "DOM",
        "MATCH_SCORE": 100.00,
        "MATCH_STATUS": (
            "DOMICILIATION_TROUVEE"
            if dom_ref
            else "MATCH_PERIODE_EXACTE"
        ),
        "NUMERO_CLIENT_RETENU": best.get("numero_client"),
        "NUMERO_DOM_RETENU": best.get("numero_dom_normalise") or best.get("numero_dom"),
        "DATE_DOM_RETENUE": best.get("date_dom"),
        "REFERENCE_EXTERNE_RETENUE": best.get("reference"),
        "REFERENCE_PREDOM_RETENUE": best.get("reference_predom"),
        "DATE_DEBUT_CONTRAT_PREDOM": best.get("date_debut_predom"),
        "DATE_FIN_CONTRAT_PREDOM": best.get("date_fin_predom"),
        "PREDOM_TROUVEE": predom_count >= 1,
    })

    # Plusieurs lignes PREDOM ne créent pas plusieurs candidats DOM,
    # mais sont signalées pour contrôle des données d'enrichissement.
    if predom_count > 1:
        result["MATCH_STATUS"] = "MATCH_DOM_TROUVE_PREDOM_MULTIPLE"

    return result

def month_segment_rows(row):
    start = parse_date(row.get("DATE_DEBUT_CONTRAT_REFERENCE"))
    end = parse_date(row.get("DATE_FIN_CONTRAT_REFERENCE"))
    if not start or not end or end < start:
        return []
    plafond = row.get("DOM_PART_TRANSFERABLE")
    rows, cursor = [], date(start.year, start.month, 1)
    while cursor <= end:
        days_month = calendar.monthrange(cursor.year, cursor.month)[1]
        month_end = date(cursor.year, cursor.month, days_month)
        seg_start, seg_end = max(start, cursor), min(end, month_end)
        if seg_start <= seg_end:
            # Règle métier :
            # - mois entièrement couvert : AAAA-MM, sans P1/P2 ;
            # - mois partiel commençant le 1er : P1 ;
            # - mois partiel commençant après le 1er : P2.
            mois_complet = (
                seg_start == cursor
                and seg_end == month_end
            )

            if mois_complet:
                part = None
                period = f"{cursor.year:04d}-{cursor.month:02d}"
            elif seg_start.day == 1:
                part = "P1"
                period = f"{cursor.year:04d}-{cursor.month:02d}P1"
            else:
                part = "P2"
                period = f"{cursor.year:04d}-{cursor.month:02d}P2"
            nb_days = (seg_end - seg_start).days + 1
            coef = round(nb_days / days_month, 8)
            rows.append({
                "FICHIER": row.get("FICHIER"),
                "NUMERO_DOMICILIATION": row.get("NUMERO_DOM_RETENU"),
                "DATE_DOMICILIATION": row.get("DATE_DOM_RETENUE"),
                "DATE_DOMICILIATION_SOURCE": "FICHIER_DOM" if row.get("DATE_DOM_RETENUE") else None,
                "NUMERO_CLIENT": row.get("NUMERO_CLIENT_RETENU"),
                "NOM_CLIENT": row.get("NOM_CLIENT_REFERENCE"),
                "NUMERO_CONTRAT": row.get("NUMERO_CONTRAT_REFERENCE"),
                "DATE_DEBUT_CONTRAT": start.isoformat(),
                "DATE_FIN_CONTRAT": end.isoformat(),
                "NUMERO_PERMIS_TRAVAIL": row.get("NUMERO_PERMIS_REFERENCE"),
                "PERIODE_TL": period, "MOIS_BASE": f"{cursor.year:04d}-{cursor.month:02d}",
                "PARTIE": part, "DATE_DEBUT_SEGMENT": seg_start.isoformat(),
                "DATE_FIN_SEGMENT": seg_end.isoformat(),
                "NB_JOURS_SEGMENT": round(float(nb_days), 2), "NB_JOURS_MOIS": round(float(days_month), 2),
                "COEFFICIENT_PRORATA": round(float(coef), 2),
                "SALAIRE_NET_REFERENCE": row.get("DOM_SALAIRE_NET_MENSUEL"),
                "TAUX_TRANSFERABLE_REFERENCE_RAW": row.get("DOM_TAUX_TRANSFERABLE"),
                "PLAFOND_MENSUEL_REFERENCE": plafond,
                "MONTANT_MAX_THEORIQUE": round(plafond * coef, 2) if plafond is not None else None,
                "MONTANT_AUTORISE_SAISI": None, "MONTANT_TRANSFERE": None,
                "SOLDE_RESTANT": None,
                "MATCH_STATUS": row.get("MATCH_STATUS"),
                "MATCH_METHOD": row.get("MATCH_METHOD"),
                "MATCH_SCORE": row.get("MATCH_SCORE"),
                "MATCH_CANDIDATES_COUNT": row.get("MATCH_CANDIDATES_COUNT"),
                "REFERENCE_PREDOM": row.get("REFERENCE_PREDOM_RETENUE"),
            })
        cursor = date(cursor.year + 1, 1, 1) if cursor.month == 12 else date(cursor.year, cursor.month + 1, 1)
    return rows

print("✅ Consolidation, matching sécurisé et planning V6.3 chargés")


In [ ]:

# Test technique P1/P2 de la V6
_demo = {
    "FICHIER": "demo.pdf",
    "DATE_DEBUT_CONTRAT_REFERENCE": "12/06/2025",
    "DATE_FIN_CONTRAT_REFERENCE": "11/06/2026",
    "DOM_PART_TRANSFERABLE": 442985.84,
    "DOM_SALAIRE_NET_MENSUEL": 466300.88,
    "DOM_TAUX_TRANSFERABLE": "95%",
    "NUMERO_CONTRAT_REFERENCE": "DEMO",
    "NOM_CLIENT_REFERENCE": "CLIENT DEMO",
    "NUMERO_PERMIS_REFERENCE": "PERMIS-DEMO",
    "NUMERO_DOM_RETENU": "DOM-DEMO",
    "DATE_DOM_RETENUE": "01/01/2025",
    "NUMERO_CLIENT_RETENU": "CLIENT-001",
    "MATCH_STATUS": "TEST",
}

_demo_rows = month_segment_rows(_demo)

assert len(_demo_rows) == 13
assert _demo_rows[0]["PERIODE_TL"] == "2025-06P2"
assert _demo_rows[-1]["PERIODE_TL"] == "2026-06P1"
assert _demo_rows[0]["NB_JOURS_MOIS"] == 30
assert _demo_rows[0]["NB_JOURS_SEGMENT"] == 19
assert _demo_rows[-1]["NB_JOURS_SEGMENT"] == 11

print("✅ Test planning P1/P2 V6 réussi")
print(
    _demo_rows[0]["PERIODE_TL"],
    _demo_rows[0]["MONTANT_MAX_THEORIQUE"],
)
print(
    _demo_rows[-1]["PERIODE_TL"],
    _demo_rows[-1]["MONTANT_MAX_THEORIQUE"],
)


## 10. Gestion des mois scindés P1 / P2 et du planning TL

In [ ]:

assert normalize_dom_reference("271901|2026.1|40|00119|DZD") == "271901202614000119DZD"
assert normalize_dom_reference("2026.1.40-00119") == "271901202614000119DZD"
assert dom_reference_is_valid("271901202614000119DZD")
print("✅ Test format DOM réussi")


### Test de la règle P1 / P2 demandée

In [ ]:

print("Le planning est généré uniquement depuis la page ENGAGEMENT_DOMICILIATION.")


## 11. Classification et extraction d’un dossier

In [ ]:

def classify_pages(pages):
    records = []

    for start_idx in range(0, len(pages), GPU_BATCH_SIZE_CLASSIFICATION):
        batch = pages[start_idx:start_idx + GPU_BATCH_SIZE_CLASSIFICATION]
        outputs = ask_batch(
            PROMPT_CLASSIFICATION,
            [item["image"] for item in batch],
            MAX_NEW_TOKENS_CLASSIFICATION,
        )

        for page, output in zip(batch, outputs):
            parsed = parse_json_response(output["text"])
            doc_type = (
                parsed.get("type_document")
                or parsed.get("type")
                or "AUTRE"
            )
            confidence = parsed.get("confidence", 0)

            try:
                confidence = float(confidence or 0)
            except Exception:
                confidence = 0.0

            if doc_type not in TYPES_VALIDES:
                doc_type = "AUTRE"

            if confidence < CLASSIFICATION_THRESHOLD:
                doc_type = "AUTRE"

            records.append({
                "page_num": page["page_num"],
                "image": page["image"],
                "width": page["width"],
                "height": page["height"],
                "white_ratio": page["white_ratio"],
                "doc_type": doc_type,
                "titre_detecte": parsed.get("titre_detecte"),
                "classification_confidence": confidence,
                "classification_raw_text": output["text"],
                "classification_tokens_in": output["tokens_in"],
                "classification_tokens_out": output["tokens_out"],
                "classification_elapsed_s": output["elapsed_s"],
                "raw_data": {},
                "extraction_status": "NON_LANCEE",
                "extraction_error": None,
                "extraction_raw_text": None,
                "extraction_tokens_in": 0,
                "extraction_tokens_out": 0,
                "extraction_elapsed_s": 0.0,
            })

    return records


def extract_classified_pages(records):
    grouped = defaultdict(list)

    for record in records:
        if record["doc_type"] in PROMPTS_EXTRACTION:
            grouped[record["doc_type"]].append(record)
        else:
            record["extraction_status"] = "NON_APPLICABLE"

    for doc_type, group in grouped.items():
        prompt = PROMPTS_EXTRACTION[doc_type]

        for start_idx in range(0, len(group), GPU_BATCH_SIZE_EXTRACTION):
            batch = group[start_idx:start_idx + GPU_BATCH_SIZE_EXTRACTION]

            try:
                outputs = ask_batch(
                    prompt,
                    [item["image"] for item in batch],
                    MAX_NEW_TOKENS_EXTRACTION,
                )

                for record, output in zip(batch, outputs):
                    parsed = parse_json_response(output["text"])
                    record["raw_data"] = clean_raw_dict(parsed)
                    record["extraction_status"] = (
                        "OK" if parsed else "JSON_VIDE"
                    )
                    record["extraction_raw_text"] = output["text"]
                    record["extraction_tokens_in"] = output["tokens_in"]
                    record["extraction_tokens_out"] = output["tokens_out"]
                    record["extraction_elapsed_s"] = output["elapsed_s"]

            except Exception as exc:
                for record in batch:
                    record["extraction_status"] = "ERREUR"
                    record["extraction_error"] = repr(exc)

    return records


def process_pdf(pdf_path, verbose=True):
    t0 = time.time()

    if verbose:
        log(f"📁 {pdf_path.name}")

    pages = pdf_to_pages(pdf_path)
    if not pages:
        raise ValueError(f"Aucune page détectée dans {pdf_path.name}")

    records = classify_pages(pages)
    records = extract_classified_pages(records)

    page_rows = [
        build_page_row(pdf_path.name, record)
        for record in records
    ]

    engagement_data = {}
    for record in records:
        if record.get("doc_type") == "ENGAGEMENT_DOMICILIATION":
            engagement_data = record.get("raw_data") or {}
            break

    dossier_row = consolidate_dossier(pdf_path.name, records)
    planning = []

    tokens_in = sum(
        record.get("classification_tokens_in", 0)
        + record.get("extraction_tokens_in", 0)
        for record in records
    )
    tokens_out = sum(
        record.get("classification_tokens_out", 0)
        + record.get("extraction_tokens_out", 0)
        for record in records
    )

    elapsed_total = round(time.time() - t0, 3)

    dossier_row.update({
        "STATUT_TRAITEMENT_PIPELINE": "TRAITE_NOUVEAU",
        "TEMPS_ECOULE_DOSSIER_S": round(elapsed_total, 2),
        "TOKENS_IN_DOSSIER": int(tokens_in),
        "TOKENS_OUT_DOSSIER": int(tokens_out),
        "TOKENS_TOTAL_DOSSIER": int(tokens_in + tokens_out),
    })

    dossier = {
        "source_file": pdf_path.name,
        "source_sha256": sha256_file(pdf_path),
        "pipeline_version": PIPELINE_VERSION,
        "stats": {
            "pages": len(pages),
            "tokens_in": tokens_in,
            "tokens_out": tokens_out,
            "tokens_total": tokens_in + tokens_out,
            "elapsed_s": elapsed_total,
        },
        "page_records": [
            {
                key: value
                for key, value in record.items()
                if key != "image"
            }
            for record in records
        ],
        "page_rows": page_rows,
        "dossier_row": dossier_row,
        "planning_tl": planning,
    }

    # Un seul JSON par PDF, sans suffixe d'empreinte.
    checkpoint = canonical_checkpoint_path(pdf_path)
    with open(checkpoint, "w", encoding="utf-8") as f:
        json.dump(
            dossier,
            f,
            ensure_ascii=False,
            indent=2,
            default=str,
        )


    return dossier


print("✅ Classification/extraction V6.5 avec temps et tokens OK")


## 12. Export Excel compatible Alteryx

In [ ]:

def ordered_columns(rows):
    preferred = [
        "FICHIER", "NB_PAGES", "TYPES_DOCUMENTS",
        "STATUT_TRAITEMENT_PIPELINE", "TEMPS_ECOULE_DOSSIER_S",
        "TOKENS_IN_DOSSIER", "TOKENS_OUT_DOSSIER", "TOKENS_TOTAL_DOSSIER",
        "REFERENCE_DOM_EXTRAITE_RAW", "REFERENCE_DOM_EXTRAITE_NORMALISEE",
        "NUMERO_DOM_RETENU", "DATE_DOM_RETENUE",
            "REFERENCE_PREDOM_RETENUE", "DATE_DEBUT_CONTRAT_PREDOM",
            "DATE_FIN_CONTRAT_PREDOM", "PREDOM_TROUVEE", "NUMERO_CLIENT_RETENU",
        "NOM_CLIENT_REFERENCE", "NUMERO_CONTRAT_REFERENCE",
        "DATE_DEBUT_CONTRAT_REFERENCE", "DATE_FIN_CONTRAT_REFERENCE",
        "NUMERO_PERMIS_REFERENCE", "MATCH_SOURCE", "MATCH_METHOD",
        "MATCH_SCORE", "MATCH_STATUS", "MATCH_CANDIDATES_COUNT",
        "REFERENCE_PREDOM_RETENUE", "DATE_DEBUT_CONTRAT_PREDOM",
        "DATE_FIN_CONTRAT_PREDOM", "PREDOM_TROUVEE",
    ]
    cols = set().union(*(r.keys() for r in rows)) if rows else set()
    return [c for c in preferred if c in cols] + sorted(cols - set(preferred))

def sheet_from_rows(wb, title, rows, columns=None, amount_columns=None):
    ws = wb.create_sheet(title)
    columns = columns or ordered_columns(rows)
    if not columns:
        ws["A1"] = "Aucune donnée"
        return
    amount_columns = set(amount_columns or [])
    fill = PatternFill("solid", fgColor="1F4E78")
    font = Font(color="FFFFFF", bold=True, name="Arial", size=9)
    for j, name in enumerate(columns, 1):
        c = ws.cell(1, j, name); c.fill = fill; c.font = font
        c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    for i, row in enumerate(rows, 2):
        for j, name in enumerate(columns, 1):
            value = row.get(name)
            if isinstance(value, (dict, list)):
                value = json.dumps(value, ensure_ascii=False)
            c = ws.cell(i, j, value)
            if name in amount_columns and isinstance(value, (int, float)):
                c.number_format = EXCEL_AMOUNT_FORMAT
    ws.freeze_panes = "A2"; ws.auto_filter.ref = ws.dimensions
    for j, name in enumerate(columns, 1):
        ws.column_dimensions[get_column_letter(j)].width = 38 if "ADRESSE" in name else min(34, max(14, len(name) + 2))

def build_long_raw_rows(dossiers):
    rows = []
    for d in dossiers:
        for page in d.get("page_records", []):
            for field, value in (page.get("raw_data") or {}).items():
                rows.append({
                    "FICHIER": d.get("source_file"), "PAGE": page.get("page_num"),
                    "TYPE_DOCUMENT": page.get("doc_type"), "CHAMP": field,
                    "VALEUR_BRUTE": value,
                    "CONFIANCE_CLASSIFICATION": page.get("classification_confidence"),
                    "STATUT_EXTRACTION": page.get("extraction_status"),
                })
    return rows

def create_excel(excel_path, dossiers, errors, reference_df=None):
    pages, dossier_rows, planning, matching = [], [], [], []
    for d in dossiers:
        pages.extend(d.get("page_rows", []))
        row = d.get("dossier_row") or {}
        dossier_rows.append(row)
        planning.extend(month_segment_rows(row))
        matching.append({k: row.get(k) for k in [
            "FICHIER", "STATUT_TRAITEMENT_PIPELINE",
            "TEMPS_ECOULE_DOSSIER_S", "TOKENS_IN_DOSSIER",
            "TOKENS_OUT_DOSSIER", "TOKENS_TOTAL_DOSSIER",
            "NOM_CLIENT_REFERENCE", "NUMERO_CONTRAT_REFERENCE",
            "DATE_DEBUT_CONTRAT_REFERENCE", "DATE_FIN_CONTRAT_REFERENCE",
            "REFERENCE_DOM_EXTRAITE_RAW", "REFERENCE_DOM_EXTRAITE_NORMALISEE",
            "MATCH_SOURCE", "MATCH_METHOD", "MATCH_SCORE", "MATCH_STATUS",
            "MATCH_CANDIDATES_COUNT", "NUMERO_CLIENT_RETENU",
            "NUMERO_DOM_RETENU", "DATE_DOM_RETENUE",
            "REFERENCE_PREDOM_RETENUE", "DATE_DEBUT_CONTRAT_PREDOM",
            "DATE_FIN_CONTRAT_PREDOM", "PREDOM_TROUVEE",
        ]})
    raw = build_long_raw_rows(dossiers)
    suivi = []
    for d in dossiers:
        stats = d.get("stats") or {}
        row = d.get("dossier_row") or {}
        suivi.append({
            "FICHIER": d.get("source_file"),
            "STATUT_TRAITEMENT_PIPELINE": row.get("STATUT_TRAITEMENT_PIPELINE"),
            "NB_PAGES": stats.get("pages"),
            "TEMPS_ECOULE_DOSSIER_S": row.get("TEMPS_ECOULE_DOSSIER_S"),
            "TOKENS_IN_DOSSIER": row.get("TOKENS_IN_DOSSIER"),
            "TOKENS_OUT_DOSSIER": row.get("TOKENS_OUT_DOSSIER"),
            "TOKENS_TOTAL_DOSSIER": row.get("TOKENS_TOTAL_DOSSIER"),
            "PIPELINE_VERSION_JSON": d.get("pipeline_version"),
            "SHA256": d.get("source_sha256"),
        })

    wb = Workbook(); wb.remove(wb.active)
    sheet_from_rows(wb, "DOSSIERS_DOMICILIATION", dossier_rows, amount_columns=AMOUNT_FIELDS)
    sheet_from_rows(wb, "SUIVI_TRAITEMENT", suivi, [
        "FICHIER", "STATUT_TRAITEMENT_PIPELINE", "NB_PAGES",
        "TEMPS_ECOULE_DOSSIER_S", "TOKENS_IN_DOSSIER",
        "TOKENS_OUT_DOSSIER", "TOKENS_TOTAL_DOSSIER",
        "PIPELINE_VERSION_JSON", "SHA256",
    ], amount_columns={"TEMPS_ECOULE_DOSSIER_S"})
    sheet_from_rows(wb, "PLANNING_TL", planning, amount_columns={
        "SALAIRE_NET_REFERENCE", "PLAFOND_MENSUEL_REFERENCE",
        "MONTANT_MAX_THEORIQUE", "MONTANT_AUTORISE_SAISI",
        "MONTANT_TRANSFERE", "SOLDE_RESTANT", "NB_JOURS_SEGMENT", "NB_JOURS_MOIS", "COEFFICIENT_PRORATA",
    })
    sheet_from_rows(wb, "PAGES_DOCUMENTS", pages)
    sheet_from_rows(wb, "EXTRACTION_BRUTE", raw, [
        "FICHIER", "PAGE", "TYPE_DOCUMENT", "CHAMP", "VALEUR_BRUTE",
        "CONFIANCE_CLASSIFICATION", "STATUT_EXTRACTION",
    ])
    sheet_from_rows(wb, "MATCHING_DOM", matching)
    sheet_from_rows(wb, "ERREURS", errors)
    if reference_df is not None:
        sheet_from_rows(wb, "REFERENTIEL_DOM_PREDOM", reference_df.where(pd.notna(reference_df), None).to_dict("records"))
    wb.save(excel_path)
    print(f"✅ Excel créé : {excel_path} | dossiers={len(dossier_rows)} | planning={len(planning)}")

print("✅ Export V6.5 chargé avec onglet SUIVI_TRAITEMENT")


In [ ]:

print(f"Nombre de PDF détectés : {len(pdfs)}")
if not pdfs:
    raise RuntimeError(
        f"Aucun PDF trouvé dans {INPUT_DIR}. "
        "Déposer les dossiers de domiciliation dans ce répertoire."
    )

diagnostic_errors = []
total_pages = 0

for p in pdfs:
    try:
        with fitz.open(str(p)) as doc:
            page_count = int(doc.page_count)
        total_pages += page_count
        print(
            f"{p.name} -> pages={page_count}, "
            f"taille={p.stat().st_size:,} octets"
        )
        if page_count <= 0:
            diagnostic_errors.append(f"{p.name}: 0 page")
    except Exception as exc:
        diagnostic_errors.append(f"{p.name}: {exc!r}")

if diagnostic_errors:
    raise RuntimeError(
        "Diagnostic PDF en erreur :\n- " + "\n- ".join(diagnostic_errors)
    )

print(f"✅ Diagnostic PDF : {len(pdfs)} fichier(s), {total_pages} page(s)")

# Nettoyage des checkpoints invalides.
# Les anciens checkpoints valides suffixés par hash seront migrés
# au moment de leur chargement vers un seul JSON canonique par PDF.
removed = 0
for json_file in JSON_DIR.glob("*.json"):
    try:
        dossier = json.loads(json_file.read_text(encoding="utf-8"))
        pages_checkpoint = int(dossier.get("stats", {}).get("pages", 0) or 0)
        records_checkpoint = dossier.get("page_records") or []
        if pages_checkpoint <= 0 or not records_checkpoint:
            json_file.unlink()
            removed += 1
            print(f"Checkpoint vide supprimé : {json_file.name}")
    except Exception:
        json_file.unlink()
        removed += 1
        print(f"Checkpoint illisible supprimé : {json_file.name}")

print(f"Checkpoints invalides supprimés : {removed}")


In [ ]:

# Test technique sur le premier PDF avant le traitement complet.
_test_pdf = pdfs[0]
_test_pages = pdf_to_pages(_test_pdf)

print(
    f"Test conversion : {_test_pdf.name} -> "
    f"{len(_test_pages)} page(s)"
)
print(
    "Dimensions première page :",
    _test_pages[0]["width"],
    "x",
    _test_pages[0]["height"],
)
print(
    "Ratio blanc première page :",
    _test_pages[0]["white_ratio"],
)

if len(_test_pages) == 0:
    raise RuntimeError("Le test de conversion PDF a retourné zéro page.")

# Libération immédiate des images du test.
del _test_pages
gc.collect()
print("✅ Test de conversion PDF réussi")


## 13. Exécution complète avec reprise automatique

In [ ]:

# Tests techniques du matching sécurisé V6.3
_test_ref = pd.DataFrame([
    {
        "numero_dom": "271901202614000119DZD",
        "numero_dom_normalise": "271901202614000119DZD",
        "date_dom": "15/01/2026",
        "numero_client": "CL001",
        "reference": "DOM-001",
        "reference_predom": "PREDOM-001",
        "date_debut_predom": "12/06/2025",
        "date_fin_predom": "11/06/2026",
        "date_debut_match": date(2025, 6, 12),
        "date_fin_match": date(2026, 6, 11),
        "DOM_MATCH_COUNT": 1,
        "PREDOM_MATCH_COUNT": 1,
    }
])

# Numéro DOM absent + période exacte unique : attribution autorisée
_test_row_unique = {
    "REFERENCE_DOM_EXTRAITE_NORMALISEE": None,
    "DATE_DEBUT_CONTRAT_REFERENCE": "12/06/2025",
    "DATE_FIN_CONTRAT_REFERENCE": "11/06/2026",
}
_test_match_unique = match_dossier(_test_row_unique, _test_ref)
assert _test_match_unique["MATCH_STATUS"] == "MATCH_PERIODE_EXACTE"
assert _test_match_unique["NUMERO_DOM_RETENU"] == "271901202614000119DZD"

# Aucun match : aucun numéro DOM ne doit être renseigné
_test_row_none = {
    "REFERENCE_DOM_EXTRAITE_NORMALISEE": None,
    "DATE_DEBUT_CONTRAT_REFERENCE": "01/01/2030",
    "DATE_FIN_CONTRAT_REFERENCE": "31/12/2030",
}
_test_match_none = match_dossier(_test_row_none, _test_ref)
assert _test_match_none["MATCH_STATUS"] == "AUCUN_MATCH_PERIODE"
assert _test_match_none["NUMERO_DOM_RETENU"] is None

# Plusieurs candidats : aucun numéro DOM ne doit être renseigné
_test_ref_multiple = pd.concat([_test_ref, _test_ref.assign(
    numero_dom="271901202624000120DZD",
    numero_dom_normalise="271901202624000120DZD",
)], ignore_index=True)
_test_match_multiple = match_dossier(_test_row_unique, _test_ref_multiple)
assert _test_match_multiple["MATCH_STATUS"] == "PLUSIEURS_CANDIDATS_PERIODE"
assert _test_match_multiple["NUMERO_DOM_RETENU"] is None

print("✅ Tests matching sécurisé V6.3 réussis")


In [ ]:

# Tests techniques V6.4 : mois complet sans P1/P2 et date DOM dans Planning_TL
_test_full_month = {
    "FICHIER": "demo.pdf",
    "NUMERO_DOM_RETENU": "271901202614000119DZD",
    "DATE_DOM_RETENUE": "15/01/2026",
    "NUMERO_CLIENT_RETENU": "CL001",
    "NOM_CLIENT_REFERENCE": "CLIENT TEST",
    "NUMERO_CONTRAT_REFERENCE": "CTR001",
    "DATE_DEBUT_CONTRAT_REFERENCE": "01/05/2026",
    "DATE_FIN_CONTRAT_REFERENCE": "31/05/2026",
    "NUMERO_PERMIS_REFERENCE": "PT001",
    "DOM_PART_TRANSFERABLE": 100000.00,
    "DOM_SALAIRE_NET_MENSUEL": 120000.00,
    "DOM_TAUX_TRANSFERABLE": "80%",
    "MATCH_STATUS": "DOMICILIATION_TROUVEE",
    "MATCH_METHOD": "NUMERO_DOM_EXACT",
    "MATCH_SCORE": 100.00,
    "MATCH_CANDIDATES_COUNT": 1,
    "REFERENCE_PREDOM_RETENUE": "PREDOM001",
}

_test_rows = month_segment_rows(_test_full_month)
assert len(_test_rows) == 1
assert _test_rows[0]["PERIODE_TL"] == "2026-05"
assert _test_rows[0]["PARTIE"] is None
assert _test_rows[0]["COEFFICIENT_PRORATA"] == 1.00
assert _test_rows[0]["DATE_DOMICILIATION"] == "15/01/2026"
assert _test_rows[0]["DATE_DOMICILIATION_SOURCE"] == "FICHIER_DOM"

print("✅ Tests V6.4 réussis : mois complet sans P1/P2 + date DOM dans Planning_TL")


In [ ]:

# Tests techniques V6.5 : un seul JSON par PDF et statistiques dossier
assert canonical_checkpoint_path(Path("07000-670909-202605-DOM.pdf")).name == \
       "07000-670909-202605-DOM.json"

_test_stats_dossier = {
    "dossier_row": {"FICHIER": "demo.pdf"},
    "stats": {
        "elapsed_s": 12.345,
        "tokens_in": 1000,
        "tokens_out": 250,
        "tokens_total": 1250,
    },
}
_test_stats_dossier = enrich_dossier_row_with_stats(
    _test_stats_dossier,
    "REPRIS_JSON_EXISTANT",
)
_test_row = _test_stats_dossier["dossier_row"]

assert _test_row["TEMPS_ECOULE_DOSSIER_S"] == 12.35
assert _test_row["TOKENS_IN_DOSSIER"] == 1000
assert _test_row["TOKENS_OUT_DOSSIER"] == 250
assert _test_row["TOKENS_TOTAL_DOSSIER"] == 1250
assert _test_row["STATUT_TRAITEMENT_PIPELINE"] == "REPRIS_JSON_EXISTANT"

print("✅ Tests V6.5 réussis : reprise JSON + temps + tokens")


In [ ]:

# Tests techniques V6.7 : suivi compact Domino
assert format_duration(65) == "01:05"
assert format_duration(3661) == "01:01:01"

print("✅ Tests V6.7 réussis : suivi compact Domino")


In [ ]:

ram_free = psutil.virtual_memory().available / 1_000_000_000
log(f"RAM libre : {ram_free:.1f} GB")
reference_df = build_reference_table()
log(f"Référentiel externe : {0 if reference_df is None else len(reference_df)} ligne(s)")

all_dossiers, errors = [], []
nb_repris = 0
nb_nouveaux = 0
pipeline_start = time.time()

print_pipeline_header(len(pdfs))

for position, pdf_path in enumerate(pdfs, 1):
    try:
        dossier = load_existing_checkpoint(pdf_path)
        skipped = dossier is not None

        if skipped:
            dossier = enrich_dossier_row_with_stats(
                dossier,
                "REPRIS_JSON_EXISTANT",
            )
            nb_repris += 1
        else:
            dossier = process_pdf(pdf_path, verbose=False)
            dossier = enrich_dossier_row_with_stats(
                dossier,
                "TRAITE_NOUVEAU",
            )
            nb_nouveaux += 1

        all_dossiers.append(dossier)

        stats = dossier.get("stats") or {}
        print_compact_progress(
            position=position,
            total=len(pdfs),
            pdf_name=pdf_path.name,
            pages=stats.get("pages", 0),
            skipped=skipped,
            tokens_in=stats.get("tokens_in", 0),
            tokens_out=stats.get("tokens_out", 0),
            elapsed_s=stats.get("elapsed_s", 0),
            pipeline_start=pipeline_start,
        )

    except Exception as exc:
        errors.append({
            "FICHIER": pdf_path.name,
            "ETAPE": "PROCESS_PDF",
            "ERREUR": repr(exc),
            "DATE": datetime.now().isoformat(timespec="seconds"),
        })

        print(
            f"[{position}/{len(pdfs)}] "
            f"{pdf_path.name} | ERREUR | {exc!r}",
            flush=True,
        )
        log(f"[{position}/{len(pdfs)}] ❌ {pdf_path.name}: {exc}")

    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

for dossier in all_dossiers:
    row = dossier.get("dossier_row") or {}
    row.update(match_dossier(row, reference_df))
    dossier["dossier_row"] = row

    # Mise à jour du JSON canonique avec le matching actuel,
    # sans relancer la classification ni l'extraction VLM.
    source_pdf = INPUT_DIR / dossier.get("source_file", "")
    if source_pdf.exists():
        canonical_checkpoint_path(source_pdf).write_text(
            json.dumps(dossier, ensure_ascii=False, indent=2, default=str),
            encoding="utf-8",
        )

create_excel(EXCEL_PATH, all_dossiers, errors, reference_df)

with open(MASTER_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump({
        "generated_at": datetime.now().isoformat(timespec="seconds"),
        "pipeline_version": PIPELINE_VERSION,
        "dossiers": all_dossiers, "errors": errors,
    }, f, ensure_ascii=False, indent=2, default=str)


elapsed_pipeline = time.time() - pipeline_start
total_tokens_in = sum(
    int((d.get("stats") or {}).get("tokens_in", 0) or 0)
    for d in all_dossiers
)
total_tokens_out = sum(
    int((d.get("stats") or {}).get("tokens_out", 0) or 0)
    for d in all_dossiers
)

print(
    f"\nTerminé | total={len(all_dossiers)} "
    f"| traités={nb_nouveaux} "
    f"| skip={nb_repris} "
    f"| erreurs={len(errors)} "
    f"| IN={total_tokens_in:,} "
    f"| OUT={total_tokens_out:,} "
    f"| durée={format_duration(elapsed_pipeline)}",
    flush=True,
)

log(f"✅ Pipeline terminé | dossiers={len(all_dossiers)} | nouveaux={nb_nouveaux} | repris={nb_repris} | erreurs={len(errors)}")



## 14. Lecture des résultats

- `domiciliations_master.json` : référentiel consolidé avec les liens de renouvellement et tout le planning.
- `DOMICILIATIONS` : une ligne par contrat/domiciliation, avec informations KYC, père, mère, montants contractuels, permis, données brutes par document et montants P1/P2.
- `PLANNING_TL` : une ligne par période de transfert autorisée. Les mois partiels sont nommés, par exemple, `2025-06P2` et `2026-06P1`.
- `PLN_MONTANT_AUTORISE_SAISI` : colonne volontairement vide destinée à être complétée/validée avant les contrôles mensuels.
- `DOCUMENTS` : classification et extraction page par page.
- `CHAMPS_SOURCE` : traçabilité longue de toutes les valeurs brutes et normalisées.
- `ERREURS` : erreurs techniques.
- `PARAMETRES` : version du modèle et règles de génération.


In [ ]:

if EXCEL_PATH.exists():
    for sheet in ["DOSSIERS_DOMICILIATION", "PLANNING_TL", "MATCHING_DOM"]:
        df = pd.read_excel(EXCEL_PATH, sheet_name=sheet)
        print(f"{sheet}: {len(df)} ligne(s)")
        display(df.head(10))
else:
    print("Le fichier Excel n'a pas encore été généré.")
